# Semana 10, Aula 01 - Simulado de Exploração e Análise de Vendas
**Base real:** Online Retail II — varejo online do Reino Unido (2009–2011)

Percorraremos as RFs do miniprojeto de forma mais enxuta.

### Onde baixar os dados brutos
Vamos realizar o carregamento dessa base de dados, porém, primeiro precisamos baixa-la usando o URL abaixo:
- Kaggle: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci

In [1]:
# Passo 1 — Importações 
import pandas as pd
import numpy as np
import re, os, json
import matplotlib.pyplot as plt
import seaborn as sns

## RF01 — Carregar o dataset de vendas

In [2]:
def carregar_dados(caminho="../data/raw/online_retail_II.csv"):
    """Lê o CSV bruto da Online Retail II."""
    df = pd.read_csv(caminho)
    print(f"{len(df)} linhas carregadas.")
    return df

df_bruto = carregar_dados()
df_bruto.head()

1067371 linhas carregadas.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## RF02 — Inspecionar e descrever os dados

In [3]:
def inspecionar_dados(df):
    print("INSPECAO INICIAL")
    print(f"Shape: {df.shape}")
    print(f"Colunas: {list(df.columns)}")
    print(f"\nTipos:\n{df.dtypes}")
    print(f"\nNulos por coluna:\n{df.isnull().sum()}")

    return df.describe().round(2)

inspecionar_dados(df_bruto)

INSPECAO INICIAL
Shape: (1067371, 8)
Colunas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Tipos:
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

Nulos por coluna:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


,Quantity,Price,Customer ID
count,1067371.00,1067371.00,824364.00
mean,9.94,4.65,15324.64
std,172.71,123.55,1697.46
min,-80995.00,-53594.36,12346.00
25%,1.00,1.25,13975.00
50%,3.00,2.10,15255.00
75%,10.00,4.15,16797.00
max,80995.00,38970.00,18287.00


## RF03 — Limpar e tratar os dados (regex + datas + nulos)
A Online Retail II vem suja: descrições/cliente nulos, **devoluções** (quantidade < 0)
e **preços ≤ 0**. Tratamos tudo e convertemos a data (InvoiceDate) para `datetime`.

In [4]:
import re
import pandas as pd

def normalizar_texto(texto):
    """
    Limpa um valor de texto:
    - troca qualquer sequencia de espacos por um unico espaco
    - remove os espacos das pontas
    """
    if pd.isna(texto):
        return texto
    texto = str(texto)
    texto = re.sub(r"\s+", " ", texto)   # colapsa espacos repetidos
    return texto.strip()                 # remove espacos das pontas

def limpar_strings(df, colunas):
    """Aplica a normalizacao de texto nas colunas indicadas."""
    df = df.copy()
    for coluna in colunas:
        df[coluna] = df[coluna].apply(normalizar_texto)
    return df

def limpar_dados(df):
    """
    Limpa o DataFrame de vendas em etapas e devolve (df_limpo, relatorio).
    O relatorio guarda quantas linhas cada etapa removeu, para mostrar o impacto.
    """
    df = df.copy()
    relatorio = {}
    relatorio["registros_iniciais"] = len(df)

    # Etapa 1: normalizar textos (espacos extras em Description e Country) 
    df = limpar_strings(df, ["Description", "Country"])

    # Etapa 2: converter a coluna de data para o tipo datetime, e remover linhas sem data valida 
    # errors="coerce" transforma datas invalidas em vazio (NaT)
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    antes = len(df)
    df = df.dropna(subset=["InvoiceDate"])      # remove linhas sem data valida
    relatorio["datas_invalidas_removidas"] = antes - len(df)

    # Etapa 3: remover linhas sem cliente ou sem descricao
    # Sem Customer ID nao da para segmentar o cliente.
    antes = len(df)
    df = df.dropna(subset=["Customer ID", "Description"])
    relatorio["nulos_removidos"] = antes - len(df)

    # Etapa 4: remover devolucoes e precos invalidos
    # Quantity <= 0  -> devolucao / cancelamento
    # Price    <= 0  -> ajuste contabil ou erro (nao é uma venda real)
    antes = len(df)
    df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
    relatorio["devolucoes_e_precos_invalidos_removidos"] = antes - len(df)

    # Etapa 5: ajustar o tipo do Customer ID
    # Estava como float por causa dos nulos; agora que nao ha mais nulos, vira int.
    df["Customer ID"] = df["Customer ID"].astype(int)

    # Resumo final
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = relatorio["registros_iniciais"] - len(df)

    print("RELATORIO DE LIMPEZA")
    for etapa, valor in relatorio.items():
        print(f"  {etapa}: {valor}")

    return df, relatorio

df_v1, relatorio = limpar_dados(df_bruto)
df_v1.to_csv("../data/processed/v1_com_outliers/vendas_v1.csv", index=False)
print("\nv1 salva em ../data/processed/v1_com_outliers/")

RELATORIO DE LIMPEZA
  registros_iniciais: 1067371
  datas_invalidas_removidas: 0
  nulos_removidos: 243007
  devolucoes_e_precos_invalidos_removidos: 18815
  registros_finais: 805549
  registros_removidos_total: 261822

v1 salva em ../data/processed/v1_com_outliers/


## RF04 — Outliers (IQR): versões v1 e v2

In [5]:
def tratar_outliers(df, colunas, fator=1.5, metodo="remover"):
    """Trata outliers numericos via IQR. metodo='remover' ou 'limitar'."""
    df = df.copy()
    for col in colunas:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf, lim_sup = q1 - fator*iqr, q3 + fator*iqr
        n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
        print(f"  {col}: {n_out} outliers (lim_inf={lim_inf:.2f}, lim_sup={lim_sup:.2f})")
        if metodo == "remover":
            df = df[(df[col] >= lim_inf) & (df[col] <= lim_sup)]
        else:
            df[col] = df[col].clip(lower=lim_inf, upper=lim_sup)
    return df

tmp = df_v1.copy(); tmp["receita_total"] = tmp["Quantity"]*tmp["Price"]
df_v2 = tratar_outliers(tmp, colunas=["Quantity", "receita_total"], metodo="remover")
df_v2 = df_v2.drop(columns=["receita_total"])
print(f"v1={len(df_v1)} | v2={len(df_v2)} | removidas={len(df_v1)-len(df_v2)}")
df_v2.to_csv("../data/processed/v2_outliers_tratado/vendas_v2.csv", index=False)
print("v2 salva.")

  Quantity: 51983 outliers (lim_inf=-13.00, lim_sup=27.00)
  receita_total: 40192 outliers (lim_inf=-15.93, lim_sup=37.88)
v1=805549 | v2=713374 | removidas=92175
v2 salva.


## RF05 — Criar colunas derivadas

Uma **coluna derivada** é uma coluna nova, calculada a partir das que já existem.
A base traz dados "crus" (quantidade, preço, data, descrição); aqui nós os
transformamos em informações prontas para análise. Sem essa etapa, o `groupby`
da RF06 (receita por mês, por categoria, etc.) não teria do que se agarrar.

Criamos quatro tipos de coluna:

**1. `receita_total` = `Quantity` × `Price`**
É a métrica central do projeto: quanto cada linha de venda gerou de dinheiro.
Multiplicamos as duas colunas elemento a elemento (o pandas faz isso na linha toda
de uma vez, sem precisar de `for`).

**2. Partes da data: `mes`, `ano`, `trimestre`, `ano_mes`**
A coluna `InvoiceDate` (já convertida para data na RF03) carrega dia, mês, ano e hora
juntos. Usando o atributo **`.dt`**, "abrimos" a data e pegamos só o pedaço que
interessa:
- `.dt.month` → o número do mês (1 a 12)
- `.dt.year` → o ano
- `.dt.quarter` → o trimestre (1 a 4); colocamos um "Q" na frente para ficar "Q1", "Q2"…
- `.dt.to_period("M")` → junta ano e mês num rótulo tipo "2010-03", útil para o
  gráfico de receita ao longo do tempo (sem misturar o mês de 2009 com o de 2011).

**3. `categoria` — derivada da descrição do produto**
A base não tem uma coluna de categoria pronta. Então criamos uma: a função
`categorizar` lê a descrição (ex.: "WHITE HANGING HEART T-LIGHT HOLDER") e procura
palavras-chave para decidir o grupo (ex.: contém "HOLDER" → Iluminação). Se nenhuma
palavra bate, a categoria vira "Outros". O `.apply(categorizar)` roda essa função
em cada linha.

**4. `faixa_receita_item` — classifica cada venda por valor**
Aqui usamos o **`np.select`**, que é um jeito enxuto de escrever um
`if / elif / else` para uma coluna inteira de uma vez:
- montamos uma lista de **condições** (receita < 20; entre 20 e 100; ≥ 100)
- e uma lista de **rótulos** na mesma ordem ("Baixo Valor", "Médio Valor", "Alto Valor")
- o `np.select` testa as condições em ordem e, para cada linha, devolve o rótulo
  correspondente; se nenhuma valer, usa o `default` ("N/D").

No fim desta etapa, cada venda passa a ter: quanto gerou, quando aconteceu, de qual
categoria é e em que faixa de valor se encaixa — tudo pronto para as análises seguintes.

In [ ]:
MAPA_CATEGORIA = {
    "Natal":      ["CHRISTMAS","XMAS","ADVENT","SANTA"],
    "Iluminacao": ["LIGHT","T-LIGHT","TLIGHT","CANDLE","HOLDER","LANTERN"],
    "Cozinha":    ["MUG","TEA","CAKE","BOTTLE","JAR","BOWL","PLATE","CUP","KITCHEN"],
    "Bolsas":     ["BAG","LUNCH BAG","BACKPACK"],
    "Decoracao":  ["HEART","FRAME","SIGN","HANGING","DECORATION","ORNAMENT"],
    "Papelaria":  ["CARD","PAPER","NOTEBOOK","PEN","PENCIL","WRAP"],
    "Brinquedos": ["TOY","GAME","PLAY","DOLL","PUZZLE"],
}
def categorizar(desc):
    """Deriva categoria a partir de palavras-chave da descricao."""
    d = str(desc).upper()
    for cat, chaves in MAPA_CATEGORIA.items():
        if any(k in d for k in chaves):
            return cat
    return "Outros"

def criar_colunas_derivadas(df):
    """Cria receita_total, componentes de data, categoria e faixa_receita_item."""
    df = df.copy()
    df["receita_total"] = df["Quantity"] * df["Price"]
    df["mes"] = df["InvoiceDate"].dt.month
    df["ano"] = df["InvoiceDate"].dt.year
    df["trimestre"] = df["InvoiceDate"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano_mes"] = df["InvoiceDate"].dt.to_period("M").astype(str)
    df["categoria"] = df["Description"].apply(categorizar)
    cond = [df["receita_total"]<20, (df["receita_total"]>=20)&(df["receita_total"]<100), df["receita_total"]>=100]
    df["faixa_receita_item"] = np.select(cond, ["Baixo Valor","Medio Valor","Alto Valor"], default="N/D")
    return df

df = criar_colunas_derivadas(df_v2)
df[["InvoiceDate","receita_total","ano_mes","trimestre","categoria","faixa_receita_item"]].head()

,InvoiceDate,receita_total,ano_mes,trimestre,categoria,faixa_receita_item
4,2009-12-01 07:45:00,30.00,2009-12,Q4,Outros,Medio Valor
6,2009-12-01 07:45:00,30.00,2009-12,Q4,Cozinha,Medio Valor
8,2009-12-01 07:46:00,30.60,2009-12,Q4,Cozinha,Medio Valor
11,2009-12-01 07:46:00,30.60,2009-12,Q4,Cozinha,Medio Valor
14,2009-12-01 09:06:00,17.85,2009-12,Q4,Outros,Baixo Valor


## RF06 — Calcular métricas agregadas (`groupby`)

Agora que cada venda tem receita, data e categoria (RF05), queremos **resumos**:
em vez de olhar 700 mil linhas, respondemos perguntas de negócio agrupando os dados.

A ferramenta é o **`groupby`**, que segue a ideia "dividir → aplicar → combinar":
1. **divide** as linhas em grupos (ex.: todas as vendas de cada mês);
2. **aplica** um cálculo em cada grupo (ex.: somar a receita);
3. **combina** os resultados numa tabela pequena.

Geramos quatro métricas:
- **por mês (`ano_mes`)** → receita, quantidade e número de vendas, para ver a evolução no tempo
- **top 5 produtos** → os campeões de receita
- **por categoria** → quais grupos de produto vendem mais
- **por país** → receita total e **ticket médio** (receita média por venda) de cada país

Usamos `.agg(nome=("coluna", "função"))` para criar várias colunas resumidas de uma
vez já com nomes claros (ex.: `receita_total=("receita_total", "sum")`).

In [7]:
def calcular_metricas(df):
    """Calcula métricas agregadas por mês, produto, categoria e país.
    Retorna um dicionário onde cada chave é uma tabela-resumo."""

    metricas = {}  # dicionário que vai guardar as quatro tabelas

    # 1) RECEITA POR MÊS 
    metricas["por_mes"] = (
        df.groupby("ano_mes")          # junta todas as vendas de cada mês (ex.: "2010-03")
          .agg(                         # calcula vários resumos de uma vez para cada mês:
              receita_total=("receita_total", "sum"),   # soma da receita do mês
              quantidade=("Quantity", "sum"),           # total de itens vendidos no mês
              n_vendas=("Invoice", "count"),            # quantas linhas de venda houve
          )
          .reset_index()               # transforma o "ano_mes" (que virou índice) de volta em coluna
          .sort_values("ano_mes")      # ordena do mês mais antigo para o mais recente
    )

    # 2) TOP 5 PRODUTOS POR RECEITA 
    metricas["top_produtos"] = (
        df.groupby("Description")["receita_total"]  # agrupa por produto e olha só a receita
          .sum()                                    # soma a receita de cada produto
          .sort_values(ascending=False)             # ordena do maior para o menor
          .head(5)                                  # pega só os 5 primeiros (o ranking)
          .reset_index()                            # volta a ser uma tabela com colunas
    )

    # 3) RECEITA POR CATEGORIA 
    metricas["por_categoria"] = (
        df.groupby("categoria")["receita_total"]    # agrupa por categoria, olhando a receita
          .sum()                                    # soma a receita de cada categoria
          .reset_index()                            # vira tabela (categoria | receita_total)
          .sort_values("receita_total", ascending=False)  # da categoria que mais vende para a que menos vende
    )

    # 4) RECEITA E TICKET MÉDIO POR PAÍS 
    metricas["por_pais"] = (
        df.groupby("Country")          # agrupa as vendas por país
          .agg(
              receita_total=("receita_total", "sum"),    # quanto cada país gerou no total
              media_ticket=("receita_total", "mean"),    # valor médio de cada venda no país
          )
          .reset_index()               # país volta a ser coluna
          .sort_values("receita_total", ascending=False) # do país que mais fatura para o que menos fatura
          .head(10)                    # mostra só os 10 maiores
    )

    # Exibição das tabelas no notebook
    for nome, tabela in metricas.items():
        # nome.upper().replace('_',' ') deixa "por_mes" -> "POR MES" (título mais legível)
        print(f"\n{nome.upper().replace('_', ' ')}")
        print(tabela.to_string(index=False))   # imprime a tabela sem a coluna de índice

    return metricas  # devolve o dicionário com as quatro tabelas


metricas = calcular_metricas(df)  # roda a função e guarda o resultado em 'metricas'


POR MES
ano_mes  receita_total  quantidade  n_vendas
2009-12     299563.060      156375     26631
2010-01     220142.202      122846     19094
2010-02     233511.896      130087     20719
2010-03     323645.651      182760     28711
2010-04     285311.742      155736     23822
2010-05     291543.710      159146     25141
2010-06     309251.650      174150     27717
2010-07     277326.140      158437     23927
2010-08     269706.460      157930     23078
2010-09     369188.341      216494     30240
2010-10     501403.650      289509     44104
2010-11     563494.212      307235     54166
2010-12     390980.940      206605     35892
2011-01     218797.710      127139     18660
2011-02     211040.360      119532     17688
2011-03     278817.600      161655     24187
2011-04     235605.121      138821     20253
2011-05     309834.380      173025     24842
2011-06     278433.000      161537     23939
2011-07     266196.301      168628     23487
2011-08     288332.560      175764     23659
2

## RF07 — Segmentar clientes por nível de gasto (`lambda`)

Segmentar é **separar os clientes em grupos** conforme o quanto gastam — uma análise clássica para o negócio saber quem são os clientes mais valiosos.

Passo a passo:
1. agrupamos por `Customer ID` e **somamos** a `receita_total` de cada cliente
   (quanto cada um gastou no total);
2. classificamos cada cliente em três faixas usando uma **função `lambda`** com
   condicional encadeado:
   - **Bronze**: gasto abaixo de R$ 1.000
   - **Prata**: de R$ 1.000 a R$ 5.000
   - **Ouro**: acima de R$ 5.000

A `lambda g: "Ouro" if g > 5000 else ("Prata" if g >= 1000 else "Bronze")` é só um
`if / elif / else` escrito em uma linha: lê-se "se gastou mais de 5000 é Ouro; senão,
se gastou 1000 ou mais é Prata; senão é Bronze". No fim, contamos quantos clientes
caíram em cada segmento.

In [8]:
def segmentar_clientes(df):
    """Soma o gasto de cada cliente e classifica em Bronze/Prata/Ouro."""
    c = df.groupby("Customer ID")["receita_total"].sum().reset_index()
    c.columns = ["cliente","total_gasto"]
    c["segmento"] = c["total_gasto"].apply(
        lambda g: "Ouro" if g > 5000 else ("Prata" if g >= 1000 else "Bronze"))
    c = c.sort_values("total_gasto", ascending=False)
    print("SEGMENTACAO (Top 10)"); print(c.head(10).to_string(index=False))
    print("\nDistribuicao:\n", c["segmento"].value_counts())
    return c

clientes = segmentar_clientes(df)

SEGMENTACAO (Top 10)
 cliente  total_gasto segmento
   14911    162411.19     Ouro
   17841     58473.29     Ouro
   13089     46769.61     Ouro
   17850     46454.60     Ouro
   14156     44237.26     Ouro
   14096     40937.96     Ouro
   12748     36391.51     Ouro
   14298     32059.24     Ouro
   13081     31561.24     Ouro
   14606     27753.57     Ouro

Distribuicao:
 segmento
Bronze    3646
Prata     1690
Ouro       303
Name: count, dtype: int64


## RF08 — Estatísticas com NumPy

Até aqui usamos o pandas. Nesta etapa descemos um nível e usamos o **NumPy**
diretamente sobre os números, para demonstrar três ideias que estão por trás de toda
análise rápida de dados. Primeiro convertemos a coluna de receita num **array NumPy**
com `.to_numpy()` (um vetor de números puro, sem os rótulos do DataFrame).

Com esse array calculamos as estatísticas descritivas — média, mediana, desvio padrão,
total e os percentis 25 e 75 — usando `np.mean`, `np.median`, `np.std`, etc.

E demonstramos três conceitos-chave:

**1. Vetorização** — operar no array inteiro de uma vez, sem `for`.
`np.mean(receitas)` percorre os 700 mil valores internamente; nós só escrevemos uma
linha. É mais simples de ler e muito mais rápido que um laço em Python.

**2. Broadcasting** — aplicar um número (escalar) a um array inteiro automaticamente.
Em `(receitas / receitas.sum()) * 100`, o `receitas.sum()` é **um único número**, mas
o NumPy o "espalha" sobre cada elemento do array, calculando de uma vez a participação
percentual de cada venda no total.

**3. Boolean indexing** — filtrar com uma máscara de Verdadeiro/Falso.
`receitas > stats["media"]` gera um array de `True`/`False` (uma resposta por venda);
somar esse array (`.sum()`) conta quantos `True` existem — ou seja, quantas vendas
ficaram **acima da média**. É o equivalente vetorizado de um `for` com `if`, porém em
uma linha.

No fim, juntamos tudo num dicionário `stats`, que será exportado em JSON na RF11.

In [9]:
def calcular_estatisticas_numpy(df):
    receitas = df["receita_total"].to_numpy()
    stats = {"media":float(np.mean(receitas)), "mediana":float(np.median(receitas)),
             "desvio_padrao":float(np.std(receitas)), "total":float(np.sum(receitas)),
             "p25":float(np.percentile(receitas,25)), "p75":float(np.percentile(receitas,75))}
    receitas_pct = (receitas/receitas.sum())*100                 # broadcasting
    stats["acima_da_media"] = int((receitas > stats["media"]).sum())  # boolean indexing
    print("ESTATISTICAS NUMPY")
    for k,v in stats.items(): print(f"  {k}: {v}")
    return stats

stats = calcular_estatisticas_numpy(df)

ESTATISTICAS NUMPY
  media: 11.343832979615183
  mediana: 10.14
  desvio_padrao: 8.318176738602254
  total: 8092395.508000001
  p25: 4.16
  p75: 16.5
  acima_da_media: 320984
